# 02 — Insights via SQL

**Projeto:** Financial Behavior Intelligence  
**Objetivo:** Responder perguntas de negócio sobre o comportamento financeiro dos usuários utilizando queries SQL estruturadas via SQLite em memória.

---

## 0. Setup

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Carregar dataset tratado (output do notebook 01)
df = pd.read_csv('../data/processed/transactions_clean.csv', parse_dates=['Date'])

# Criar banco SQLite em memória e registrar a tabela
conn = sqlite3.connect(':memory:')
df.to_sql('transactions', conn, index=False, if_exists='replace')
print('Banco criado. Tabela: transactions')
print('Colunas:', df.columns.tolist())

## Função auxiliar para executar queries

In [ ]:
def query(sql, title=''):
    """Executa uma query SQL e retorna um DataFrame."""
    result = pd.read_sql_query(sql, conn)
    if title:
        print(f'--- {title} ---')
    return result

---
## Q1 — Saldo Mensal por Usuário

> Qual o saldo (receita - despesa) de cada usuário em cada mês?

In [ ]:
q1 = query("""
    SELECT
        User_ID,
        strftime('%Y-%m', Date) AS Periodo,
        SUM(CASE WHEN "Transaction Type" = 'credit' THEN Amount ELSE 0 END) AS Receita,
        SUM(CASE WHEN "Transaction Type" = 'debit'  THEN Amount ELSE 0 END) AS Despesa,
        SUM(CASE WHEN "Transaction Type" = 'credit' THEN Amount ELSE 0 END) -
        SUM(CASE WHEN "Transaction Type" = 'debit'  THEN Amount ELSE 0 END) AS Saldo
    FROM transactions
    GROUP BY User_ID, Periodo
    ORDER BY User_ID, Periodo
""", 'Saldo Mensal por Usuário')

q1.head(12)

## Q2 — Variação de Despesas Mês a Mês (MoM%)

> Quanto as despesas variaram em relação ao mês anterior, por usuário?

In [ ]:
# ADD: usar LAG() via pandas após trazer q1, ou calcular com pct_change()
q1_pivot = q1.pivot(index='Periodo', columns='User_ID', values='Despesa')
mom = q1_pivot.pct_change().mul(100).round(2)
print('Variação MoM% de Despesas (primeiros usuários):')
mom.iloc[:6, :5]

## Q3 — Ranking de Categorias por Usuário

> Quais categorias consomem mais dinheiro de cada usuário?

In [ ]:
q3 = query("""
    SELECT
        User_ID,
        Category,
        ROUND(SUM(Amount), 2) AS Total_Gasto,
        COUNT(*) AS Qtd_Transacoes
    FROM transactions
    WHERE "Transaction Type" = 'debit'
    GROUP BY User_ID, Category
    ORDER BY User_ID, Total_Gasto DESC
""", 'Ranking de Categorias por Usuário')

q3.head(15)

## Q4 — Usuários com Saldo Negativo Recorrente

> Quais usuários apresentam saldo negativo em mais de 50% dos meses? (candidatos ao perfil 'debtor')

In [ ]:
q4 = query("""
    WITH saldo_mensal AS (
        SELECT
            User_ID,
            strftime('%Y-%m', Date) AS Periodo,
            SUM(CASE WHEN "Transaction Type" = 'credit' THEN Amount ELSE -Amount END) AS Saldo
        FROM transactions
        GROUP BY User_ID, Periodo
    ),
    contagem AS (
        SELECT
            User_ID,
            COUNT(*) AS Total_Meses,
            SUM(CASE WHEN Saldo < 0 THEN 1 ELSE 0 END) AS Meses_Negativos
        FROM saldo_mensal
        GROUP BY User_ID
    )
    SELECT
        User_ID,
        Total_Meses,
        Meses_Negativos,
        ROUND(100.0 * Meses_Negativos / Total_Meses, 1) AS Pct_Negativo
    FROM contagem
    WHERE Pct_Negativo > 50
    ORDER BY Pct_Negativo DESC
""", 'Usuários com Saldo Negativo > 50% dos meses')

q4

## Q5 — Taxa de Poupança Média por Usuário

> Taxa = (Receita - Despesa) / Receita

In [ ]:
q5 = query("""
    WITH base AS (
        SELECT
            User_ID,
            strftime('%Y-%m', Date) AS Periodo,
            SUM(CASE WHEN "Transaction Type" = 'credit' THEN Amount ELSE 0 END) AS Receita,
            SUM(CASE WHEN "Transaction Type" = 'debit'  THEN Amount ELSE 0 END) AS Despesa
        FROM transactions
        GROUP BY User_ID, Periodo
    )
    SELECT
        User_ID,
        ROUND(AVG(CASE WHEN Receita > 0 THEN (Receita - Despesa) / Receita ELSE NULL END), 3) AS Taxa_Poupanca_Media
    FROM base
    GROUP BY User_ID
    ORDER BY Taxa_Poupanca_Media DESC
""", 'Taxa de Poupança Média por Usuário')

q5

## Q6 — ADD: Sua query aqui

> Descreva a pergunta de negócio que esta query responde.

In [ ]:
# ADD

## Fechar conexão

In [ ]:
conn.close()
print('Conexão encerrada.')